# Solution F1 · Five implementations compared: speed and quality as the data grows

**Type**: Compare

## Objective

Compare `GradientBoostingClassifier` (classic scikit-learn), `HistGradientBoostingClassifier`, `XGBClassifier` with `tree_method="hist"` and `"exact"` and `RandomForestClassifier` on three training set sizes (2000, 8000, 32000 examples): how much the time grows, what the AUC is worth and what matters more (algorithm or amount of data).

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import time

import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score

Xb, yb = make_classification(40000, n_features=20, n_informative=8, class_sep=0.7, flip_y=0.05, random_state=1)
X_te, y_te = Xb[-8000:], yb[-8000:]
sizes = [2000, 8000, 32000]
models = {
    "GradientBoosting": lambda: GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=0),
    "HistGradientBoosting": lambda: HistGradientBoostingClassifier(max_iter=100, max_depth=4, early_stopping=False, random_state=0),
    "XGBoost hist": lambda: xgb.XGBClassifier(n_estimators=100, max_depth=4, tree_method="hist", eval_metric="logloss", n_jobs=-1, random_state=0),
    "XGBoost exact": lambda: xgb.XGBClassifier(n_estimators=100, max_depth=4, tree_method="exact", eval_metric="logloss", n_jobs=-1, random_state=0),
    "RandomForest": lambda: RandomForestClassifier(100, n_jobs=-1, random_state=0),
}

## Commented solution

In [ ]:
index = pd.MultiIndex.from_product([list(models), sizes], names=["method", "n"])
table = pd.DataFrame(index=index, columns=["seconds", "auc"], dtype=float)
for name, create in models.items():
    for n in sizes:
        X_tr, y_tr = Xb[:n], yb[:n]
        t0 = time.perf_counter()
        m = create().fit(X_tr, y_tr)
        seconds = time.perf_counter() - t0
        table.loc[(name, n)] = [seconds, roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])]
table.round(3)

In [ ]:
seconds = table["seconds"].unstack().loc[list(models)]
auc = table["auc"].unstack().loc[list(models)]
time_growth = seconds[32000] / seconds[2000]
gb_speedup = float(seconds.loc["GradientBoosting", 32000] / seconds.loc["XGBoost hist", 32000])
print(seconds.round(2).to_string())
print("\ntime growth from 2000 to 32000 examples:", time_growth.round(1).to_dict())
print(f"at 32000 examples GradientBoosting is {gb_speedup:.0f} times slower than XGBoost hist")

In [ ]:
auc_spread_32000 = float(auc[32000].max() - auc[32000].min())
data_gain = auc[32000] - auc[2000]
scalability_conclusion = "classic gb does not scale"
matters_more = "data" if data_gain.mean() > auc_spread_32000 else "algorithm"
print(auc.round(3).to_string())
print(f"\ngap between methods at 32000 examples: {auc_spread_32000:.3f} | mean gain going from 2000 to 32000 examples: {data_gain.mean():.3f} -> matters more: {matters_more}")

## Check

In [ ]:
with verify():
    check(table.notna().all().all() and table.shape == (15, 2) and (table["auc"] > 0.88).all(), "fifteen complete measurements, AUC always over 0.88", "time with time.perf_counter around fit")
    check(time_growth is not None and time_growth["GradientBoosting"] > 10 and time_growth["XGBoost hist"] < 6 and time_growth["HistGradientBoosting"] < 6, "the classic boosting grows by over 10 times from 2000 to 32000 examples, the histogram versions by less than 6", "seconds(32000) / seconds(2000)")
    check(gb_speedup is not None and gb_speedup > 15, "at 32000 examples the classic boosting is over 15 times slower than XGBoost hist", "GB time / XGBoost hist time")
    check(auc_spread_32000 is not None and auc_spread_32000 < 0.02 and (data_gain > 0.02).all(), "at 32000 examples the five methods differ by less than 2 points of AUC, while going from 2000 to 32000 examples gains over 2 points for each", "maximum AUC - minimum AUC; auc(32000) - auc(2000)")
    check(scalability_conclusion == "classic gb does not scale" and matters_more == "data", "the classic boosting does not scale and the factor that matters most is the amount of data", "compare 0.011 with 0.03")
done()

## What you learned

- **The classic scikit-learn boosting does not scale:** 1.6 s at 2000 examples, 30 s at 32000 (19 times, for 16 times the data) against 0.3-0.4 s for XGBoost `hist` and about 1 s for `HistGradientBoosting`. At 32000 examples it is 70 times slower than XGBoost. The difference is the threshold search algorithm: exact (it sorts the features at every tree) against histogram (256 bins built once).
- **`XGBoost exact` also scales badly:** from 0.3 s to 2.8 s (about 8 times). The histogram method is the default recommended by XGBoost 2 onwards, and for large data it is the only way.
- **The quality is almost the same:** at 32000 examples the five methods are within 1 point of AUC (0.941-0.952), while going from 2000 to 32000 examples is worth 3-4 points for all of them. With enough data the choice of the algorithm moves less than the amount of data or the feature engineering.
- **Practical choice:** `HistGradientBoosting` if you want to stay in scikit-learn (no dependencies, it handles NaN and categoricals); `XGBoost` (or LightGBM) if you need refined early stopping, monotonic constraints, GPU, `pred_contribs`; `GradientBoostingClassifier` only for teaching experiments or very small datasets.

## Common pitfall

Comparing algorithms on a small dataset and concluding "they are all the same in speed": the differences of scale emerge only as `n` grows. Always measure at several sizes.

## Interview question

> *Click on a question to show its answer.*

<details><summary>Why is histogram boosting much faster than the classic one?</summary>

It quantizes each feature into at most 256 bins once and then, for each tree, searches the threshold only among the bins by building histograms of gradients: the cost per tree is linear in n and independent of the number of distinct values. The exact method has to sort or scan all the values at every tree. The quality changes very little because 256
candidate thresholds are almost always enough.

</details>